# M4.10 · Segmentasyon — FCN, transposed convolution, U-Net

Her pikseli sınıflandırmak: anlamsal (semantic) ve örnek (instance) segmentasyon. CNN'in toplam adımı (32) çözünürlüğü yok eder; FCN bunu bir yukarı örnekleme katmanıyla geri kazanır. Transposed convolution'un dört eşdeğer görünümü: girdiyi çekirdekle "saçma" (scatter), sıfır ekleyip normal konvolüsyon (kesirli adım), konvolüsyon matrisinin transpozu ve konvolüsyonun girdiye göre gradyanı. Bilineer çekirdekle başlatma, FCN-32s → FCN-8s atlama bağlantıları, U-Net'in kodlayıcı-kod çözücüsü ve kopyalama bağlantıları, valid konvolüsyonlu 572 → 388 geometrisi, IoU/Dice/mIoU, dama tahtası (checkerboard) artefaktı, alt-piksel konvolüsyonu (PixelShuffle), genişletilmiş (dilated) konvolüsyon ve bağlantılı bileşenlerle örnek ayırma.

Etkileşimli sürüm: [https://m-gonul.github.io/pytorch-lab/m4-10-segmentation.html](https://m-gonul.github.io/pytorch-lab/m4-10-segmentation.html)

**Nasıl çalışılır:** her aşamada önce `%%writefile` hücresindeki kodu düzenleyip çalıştır (dosyaya yazar), sonra altındaki **Kontrol** hücresini çalıştır. Kontroller sitedekiyle birebir aynı. GPU gerekmez; Faz 7 modülleri hariç.

In [ ]:
#@title Kurulum: kontrol kütüphanesi (çalıştır, düzenleme)
%%writefile labcheck.py
"""labcheck — pytorch-lab'in ortak kontrol çekirdeği.

Aynı dosya üç yerde birebir çalışır:
  * tarayıcıda (Pyodide, Web Worker içinde),
  * yerel runner'da (uv run runner/torchlab.py),
  * CI'da ve Colab notebook'larında.

Protokol: öğrenci kodu `exec` ile ayrı bir namespace'te çalıştırılır, ardından
aşamanın kontrol dosyası (`check(ns, R)` fonksiyonu tanımlar) aynı namespace
üzerinde çalışır. Sonuç JSON'a çevrilebilir bir dict olarak döner.
"""
from __future__ import annotations

import ast
import contextlib
import io
import json
import math
import time
import traceback

__all__ = ["Report", "run_stage", "close", "to_list", "no_loops", "no_imports",
           "uses_any", "need", "ensure", "bench", "fmt_time", "loop_depth", "max_loops"]


# --------------------------------------------------------------------------- yardımcılar
def to_list(x):
    """Tensor / ndarray / liste / skaleri iç içe Python listesine (float) çevirir."""
    if hasattr(x, "detach"):  # torch.Tensor
        x = x.detach().cpu()
        x = x.tolist()
    elif hasattr(x, "tolist"):  # numpy
        x = x.tolist()
    if isinstance(x, (list, tuple)):
        return [to_list(v) for v in x]
    if isinstance(x, bool):
        return float(x)
    if isinstance(x, (int, float)):
        return float(x)
    return x


def _shape(x):
    if isinstance(x, list):
        return (len(x),) + (_shape(x[0]) if x else ())
    return ()


def _flat(x):
    if isinstance(x, list):
        for v in x:
            yield from _flat(v)
    else:
        yield x


def close(got, want, rtol=1e-5, atol=1e-6, what="sonuç"):
    """`got` ile `want` aynı shape'te ve sayısal olarak yakın mı? Değilse AssertionError."""
    g, w = to_list(got), to_list(want)
    if _shape(g) != _shape(w):
        raise AssertionError(f"{what}: shape {_shape(g)} bekleniyordu {_shape(w)}")
    worst = 0.0
    for a, b in zip(_flat(g), _flat(w)):
        if not isinstance(a, float):
            raise AssertionError(f"{what}: sayı olmayan değer {a!r}")
        if math.isnan(a) and math.isnan(b):
            continue
        err = abs(a - b)
        if err > atol + rtol * abs(b):
            raise AssertionError(f"{what}: {a:.6g} bulundu, {b:.6g} bekleniyordu")
        worst = max(worst, err)
    return worst


def _tree(src):
    try:
        return ast.parse(src)
    except SyntaxError:
        return None


def no_loops(src, allow_comprehension=False):
    """Kaynakta `for`/`while` döngüsü (ve istenirse comprehension) var mı? Varsa AssertionError."""
    t = _tree(src)
    if t is None:
        return
    banned = (ast.For, ast.While, ast.AsyncFor)
    if not allow_comprehension:
        banned += (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)
    for node in ast.walk(t):
        if isinstance(node, banned):
            kind = "döngü" if isinstance(node, (ast.For, ast.While, ast.AsyncFor)) else "comprehension"
            raise AssertionError(f"satır {node.lineno}: bu aşamada {kind} yasak — işlemi vektörel yaz")


def _func(tree, name):
    for node in ast.walk(tree):
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) and node.name == name:
            return node
    return None


def loop_depth(src, func=None):
    """Kaynakta (ya da yalnız `func` fonksiyonunda) iç içe for/while derinliği. Comprehension'lar da sayılır."""
    t = _tree(src)
    if t is None:
        return 0
    if func is not None:
        t = _func(t, func)
        if t is None:
            raise AssertionError(f"`{func}` fonksiyonu bulunamadı")
    loops = (ast.For, ast.While, ast.AsyncFor, ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)

    def depth(node):
        own = 1 if isinstance(node, loops) else 0
        if isinstance(node, (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)):
            own = len(node.generators)
        return own + max((depth(c) for c in ast.iter_child_nodes(node)), default=0)
    return depth(t)


def max_loops(src, n, func=None, why=""):
    """Döngü derinliği en fazla n mi? (ör. eğitim adımları için tek döngü serbest, örnekler üzerinde döngü yasak)"""
    d = loop_depth(src, func)
    if d > n:
        where = f"`{func}` içinde " if func else ""
        raise AssertionError(f"{where}{d} katmanlı döngü var; en fazla {n} olmalı{(' — ' + why) if why else ''}")


def no_imports(src, names):
    """Belirtilen modüllerin import edilmediğini doğrular (ör. b aşamasında numpy yok)."""
    t = _tree(src)
    if t is None:
        return
    for node in ast.walk(t):
        mods = []
        if isinstance(node, ast.Import):
            mods = [a.name.split(".")[0] for a in node.names]
        elif isinstance(node, ast.ImportFrom) and node.module:
            mods = [node.module.split(".")[0]]
        for m in mods:
            if m in names:
                raise AssertionError(f"satır {node.lineno}: bu aşamada `{m}` kullanılmamalı")


def uses_any(src, words):
    """Kaynak, verilen isimlerden en az birini (attribute ya da isim olarak) kullanıyor mu?"""
    t = _tree(src)
    if t is None:
        return False
    found = set()
    for node in ast.walk(t):
        if isinstance(node, ast.Attribute):
            found.add(node.attr)
        elif isinstance(node, ast.Name):
            found.add(node.id)
    return any(w in found for w in words)


def ensure(cond, msg):
    """`cond` yanlışsa `msg` ile AssertionError."""
    if not cond:
        raise AssertionError(msg)


def need(ns, name, kind="fonksiyon"):
    if name not in ns:
        raise AssertionError(f"`{name}` adlı {kind} tanımlanmamış")
    return ns[name]


def fmt_time(s):
    if s < 1e-6:
        return f"{s * 1e9:.0f} ns"
    if s < 1e-3:
        return f"{s * 1e6:.1f} µs"
    if s < 1:
        return f"{s * 1e3:.2f} ms"
    return f"{s:.2f} s"


def bench(fn, repeat=5, number=None, budget=0.25):
    """`fn()`'in medyan süresi (saniye). number verilmezse ~budget saniyeye göre ayarlanır."""
    fn()  # ısınma (lazy init, torch.compile vb.)
    if number is None:
        t0 = time.perf_counter()
        fn()
        one = max(time.perf_counter() - t0, 1e-7)
        number = max(1, min(10_000, int(budget / one / repeat)))
    times = []
    for _ in range(repeat):
        t0 = time.perf_counter()
        for _ in range(number):
            fn()
        times.append((time.perf_counter() - t0) / number)
    times.sort()
    return times[len(times) // 2]


# --------------------------------------------------------------------------- rapor
class Report:
    def __init__(self):
        self.items = []    # {"name", "ok", "msg"}
        self.tables = []   # {"title", "cols", "rows", "note"}
        self.notes = []

    def test(self, name, fn=None):
        """`R.test("ad", lambda: ...)` ya da dekoratör olarak `@R.test("ad")`."""
        if fn is None:
            return lambda f: (self.test(name, f), f)[1]
        try:
            res = fn()
            msg = res if isinstance(res, str) else ""
            self.items.append({"name": name, "ok": True, "msg": msg})
        except AssertionError as e:
            self.items.append({"name": name, "ok": False, "msg": str(e) or "koşul sağlanmadı"})
        except Exception as e:  # öğrenci kodu patladı
            tb = traceback.extract_tb(e.__traceback__)
            where = next((f"satır {f.lineno}: " for f in reversed(tb) if f.filename == "<öğrenci>"), "")
            self.items.append({"name": name, "ok": False, "msg": f"{where}{type(e).__name__}: {e}"})
        return fn

    def table(self, title, cols, rows, note=""):
        self.tables.append({"title": title, "cols": list(cols), "rows": [list(map(str, r)) for r in rows], "note": note})

    def note(self, text):
        self.notes.append(str(text))

    @property
    def ok(self):
        return bool(self.items) and all(i["ok"] for i in self.items)


def _short_tb(e):
    tb = traceback.extract_tb(e.__traceback__)
    frames = [f for f in tb if f.filename == "<öğrenci>"]
    loc = f"satır {frames[-1].lineno}: " if frames else ""
    return f"{loc}{type(e).__name__}: {e}"


def run_stage(code, check_code=None, max_out=20_000):
    """Öğrenci kodunu çalıştırır, istenirse kontrolleri uygular. JSON'a çevrilebilir dict döner."""
    out = io.StringIO()
    ns = {"__name__": "__main__", "__source__": code}
    t0 = time.perf_counter()
    error = None
    with contextlib.redirect_stdout(out), contextlib.redirect_stderr(out):
        try:
            exec(compile(code, "<öğrenci>", "exec"), ns)
        except SyntaxError as e:
            error = f"satır {e.lineno}: SyntaxError: {e.msg}"
        except Exception as e:
            error = _short_tb(e)
    elapsed = time.perf_counter() - t0
    R = Report()
    if check_code is not None and error is None:
        cns = {"__name__": "labcheck_stage"}
        with contextlib.redirect_stdout(out):
            try:
                exec(compile(check_code, "<kontrol>", "exec"), cns)
                cns["check"](ns, R)
            except Exception as e:  # kontrol dosyası hatası: kullanıcıya göster
                R.items.append({"name": "kontrol çalıştırılamadı", "ok": False,
                                "msg": f"{type(e).__name__}: {e}"})
    text = out.getvalue()
    if len(text) > max_out:
        text = text[:max_out] + f"\n… (çıktı {len(text) - max_out} karakter kısaltıldı)"
    return {
        "stdout": text,
        "error": error,
        "elapsed": elapsed,
        "checked": check_code is not None,
        "ok": error is None and (check_code is None or R.ok),
        "items": R.items,
        "tables": R.tables,
        "notes": R.notes,
    }


def run_stage_json(code, check_code=None):
    return json.dumps(run_stage(code, check_code), ensure_ascii=False)


def run_file(path, check_code):
    """Colab/Jupyter yardımcısı: `%%writefile` ile yazılmış aşama dosyasını kontrol eder ve özetler."""
    with open(path, encoding="utf-8") as f:
        res = run_stage(f.read(), check_code)
    if res["stdout"].strip():
        print(res["stdout"].rstrip())
    if res["error"]:
        print("✗ hata:", res["error"])
    for it in res["items"]:
        print(("✓ " if it["ok"] else "✗ ") + it["name"] + (f" — {it['msg']}" if it["msg"] else ""))
    for t in res["tables"]:
        print(f"\n{t['title']}")
        w = [max(len(str(c)), *(len(r[i]) for r in t["rows"])) for i, c in enumerate(t["cols"])]
        print("  ".join(str(c).ljust(w[i]) for i, c in enumerate(t["cols"])))
        for r in t["rows"]:
            print("  ".join(v.ljust(w[i]) for i, v in enumerate(r)))
        if t["note"]:
            print(t["note"])
    print("\nSONUÇ:", "geçti ✓" if res["ok"] else "henüz değil ✗")
    return res["ok"]

In [ ]:
#@title Kurulum: bu modülün kontrolleri (çalıştır, düzenleme)
%%writefile checks_m4_10_segmentation.py
CHECKS = {
 "b": "from labcheck import close, no_imports, need, ensure\n\n\ndef check(ns, R_):\n    src = ns[\"__source__\"]\n    R_.test(\"numpy kullanılmıyor\", lambda: no_imports(src, {\"numpy\", \"torch\"}))\n\n    def t():\n        ensure(need(ns, \"tconv_out_size\")(7, 4, 2, 1) == 14 and ns[\"tconv_out_size\"](7, 3, 2, 1, 1) == 14, \"çıkış boyutu (output_padding dahil)\")\n        close(need(ns, \"conv_transpose1d\")([1, 2, 3], [1, 2, 1], 2), [1, 2, 3, 4, 5, 6, 3], what=\"1B scatter\")\n        close(ns[\"conv_transpose1d\"]([1, 2, 3], [0.5, 1, 0.5], 2, 0)[:5], [0.5, 1, 1.5, 2, 2.5], what=\"[½, 1, ½] = doğrusal interpolasyon\")\n        close(ns[\"conv_transpose1d\"]([1, 2, 3], [1, 2, 1], 2, 1), [2, 3, 4, 5, 6], what=\"dolgu kırpması\")\n        y = need(ns, \"conv_transpose2d\")([[1, 2], [3, 4]], [[1, 1], [1, 1]], 2)\n        close(y, [[1, 1, 2, 2], [1, 1, 2, 2], [3, 3, 4, 4], [3, 3, 4, 4]], what=\"k = s = 2: en yakın komşu büyütme\")\n        z = ns[\"conv_transpose2d\"]([[1, 0], [0, 0]], [[1, 2, 3], [4, 5, 6], [7, 8, 9]], 1)\n        ensure(z[1][1] == 5 and len(z) == 4, \"tek piksel → çekirdeğin kopyası\")\n        ensure(need(ns, \"unet_sizes\")(572)[-1] == 388 and ns[\"unet_sizes\"](572)[4] == 32, \"U-Net 572 → 388\")\n        d, i = need(ns, \"dice_iou\")([[1, 1, 1], [1, 1, 1], [0, 0, 0]], [[0, 1, 1], [0, 1, 1], [0, 1, 0]])\n        close((d, i), (8 / 11, 4 / 7), what=\"Dice ve IoU\")\n        rows = []\n        for k, s in ((2, 2), (3, 2), (4, 2), (3, 1), (5, 3), (6, 3)):\n            c = need(ns, \"overlap_counts\")(12, k, s)[k:-k]\n            rows.append([k, s, \"\".join(str(v) for v in c[:8]), \"düzgün\" if min(c) == max(c) else \"dama tahtası\"])\n        R_.table(\"1B transposed conv: iç bölgede çıkış başına katkı sayısı\", [\"k\", \"s\", \"desen\", \"\"], rows,\n                 note=\"k, s'nin katı olmadığında katkı sayısı periyodik değişir: aynı ağırlıklar bile 'çizgili' çıktı üretir (2B'de dama tahtası). Öğrenme bunu telafi edebilir ama genellikle tamamen etmez.\")\n    R_.test(\"transposed conv ve ölçütler\", t)\n",
 "c": "import numpy as np\nfrom labcheck import close, max_loops, need, ensure\n\n\ndef check(ns, R_):\n    src = ns[\"__source__\"]\n    R_.test(\"döngüsüz\", lambda: [max_loops(src, 0, func=f) for f in (\"conv_matrix\", \"tconv2d\", \"tconv_zero_insert\", \"bilinear_kernel\", \"confusion\", \"miou\")])\n    rng = np.random.default_rng(0)\n\n    def t_eq():\n        for (H, k, s, p) in ((5, 3, 1, 0), (6, 3, 2, 1), (5, 4, 2, 1), (7, 5, 3, 2)):\n            w = rng.normal(size=(k, k))\n            M = need(ns, \"conv_matrix\")(w, H, s, p)\n            x = rng.normal(size=(H, H))\n            xp = np.pad(x, p)\n            Ho = (H + 2 * p - k) // s + 1\n            ref = np.array([[(xp[i * s:i * s + k, j * s:j * s + k] * w).sum() for j in range(Ho)] for i in range(Ho)])\n            close(M @ x.ravel(), ref.ravel(), atol=1e-10, what=f\"C·x = konv. (H={H}, k={k}, s={s}, p={p})\")\n            y = rng.normal(size=(Ho, Ho))\n            T = (Ho - 1) * s + k - 2 * p\n            close(need(ns, \"tconv2d\")(y[None], w[None, None], s, p)[0], (M.T @ y.ravel()).reshape(H, H)[:T, :T], atol=1e-10, what=\"transposed conv = Cᵀ·y\")\n        x = rng.normal(size=(3, 5, 6))\n        w = rng.normal(size=(3, 4, 3, 3))\n        for s, p in ((1, 0), (2, 0), (2, 1), (3, 2)):\n            a = ns[\"tconv2d\"](x, w, s, p)\n            b = need(ns, \"tconv_zero_insert\")(x, w, s, p)\n            ensure(a.shape == (4, (5 - 1) * s + 3 - 2 * p, (6 - 1) * s + 3 - 2 * p), \"çıkış şekli\")\n            close(a, b, atol=1e-10, what=f\"scatter = sıfır ekleme + konv. (s={s}, p={p})\")\n    R_.test(\"dört görünüm\", t_eq)\n\n    def t_bil():\n        K = need(ns, \"bilinear_kernel\")(2)\n        close(K[1, 1], 0.5625, what=\"merkez\")\n        ramp = np.add.outer(np.arange(8.), 2 * np.arange(8.))[None]\n        for f in (2, 4):\n            Kf = ns[\"bilinear_kernel\"](f)\n            k = Kf.shape[0]\n            up = ns[\"tconv2d\"](ramp, Kf[None, None], f, (k - f) // 2 if k % 2 == 0 else (k - f + 1) // 2)\n            inner = up[0, f:-f, f:-f]\n            dr, dc = np.diff(inner, axis=0), np.diff(inner, axis=1)\n            ensure(np.allclose(dr, dr[0, 0]) and np.allclose(dc, dc[0, 0]), f\"×{f} bilineer: doğrusal rampa doğrusal kalır\")\n            close(dc[0, 0] * f, 2.0, what=\"eğim korunur\")\n        pred = rng.integers(0, 3, (20, 20))\n        gt = rng.integers(0, 3, (20, 20))\n        cm = need(ns, \"confusion\")(pred, gt, 3)\n        ensure(cm.sum() == 400 and cm[1, 2] == ((gt == 1) & (pred == 2)).sum(), \"karışıklık matrisi (satır gerçek)\")\n        iou, m = need(ns, \"miou\")(cm)\n        close(iou[0], ((gt == 0) & (pred == 0)).sum() / ((gt == 0) | (pred == 0)).sum(), what=\"sınıf IoU\")\n    R_.test(\"bilineer çekirdek ve mIoU\", t_bil)\n",
 "d": "import time\nimport torch\nimport torch.nn.functional as F\nfrom labcheck import close, need, ensure\n\n\ndef check(ns, R_):\n    def t_adj():\n        torch.manual_seed(0)\n        for s, p, H in ((1, 0, 6), (2, 1, 8), (2, 0, 9), (3, 1, 10)):\n            w = torch.randn(4, 3, 3, 3)\n            Ho = (H + 2 * p - 3) // s + 1\n            y = torch.randn(2, 4, Ho, Ho)\n            g = need(ns, \"conv_input_grad\")(y, w, s, p, H)\n            op = H - ((Ho - 1) * s - 2 * p + 3)\n            close(g.numpy(), F.conv_transpose2d(y, w, stride=s, padding=p, output_padding=op).numpy(), atol=1e-5, what=f\"∂conv/∂x = conv_transpose (s={s}, p={p})\")\n        t = need(ns, \"bilinear_init_\")(torch.nn.ConvTranspose2d(2, 2, 4, 2, 1))\n        x = torch.rand(1, 2, 6, 6)\n        close(t(x)[:, :, 1:-1, 1:-1].detach().numpy(), F.interpolate(x, scale_factor=2, mode=\"bilinear\", align_corners=False)[:, :, 1:-1, 1:-1].numpy(), atol=1e-5, what=\"bilineer başlatılmış tconv = F.interpolate (iç bölge)\")\n    R_.test(\"transposed conv = adjoint; bilineer başlatma\", t_adj)\n\n    def t_train():\n        X, Y = need(ns, \"seg_dataset\")(1000, 0)\n        Xt, Yt = ns[\"seg_dataset\"](200, 1)\n        rows, res = [], {}\n        t00 = time.perf_counter()\n        for name, mk in ((\"FCN adım 8, rastgele tconv\", lambda: ns[\"FCN\"](4, 12, False, False)), (\"FCN adım 8, bilineer tconv\", lambda: ns[\"FCN\"](4, 12, False, True)),\n                         (\"FCN-8s düzeni (atlamalı)\", lambda: ns[\"FCN\"](4, 12, True, True)), (\"U-Net\", lambda: need(ns, \"UNet\")(4, 12))):\n            torch.manual_seed(0)\n            m = mk()\n            ensure(tuple(m(X[:2]).shape) == (2, 4, 64, 64), f\"{name}: çıkış (N, C, 64, 64)\")\n            opt = torch.optim.Adam(m.parameters(), 3e-3)\n            for ep in range(5):\n                m.train()\n                perm = torch.randperm(len(X))\n                for i in range(0, len(X), 32):\n                    idx = perm[i:i + 32]\n                    loss = F.cross_entropy(m(X[idx]), Y[idx])\n                    opt.zero_grad()\n                    loss.backward()\n                    opt.step()\n            m.eval()\n            with torch.no_grad():\n                P = m(Xt).argmax(1)\n            res[name] = (need(ns, \"miou\")(P, Yt, 4), need(ns, \"edge_acc\")(P, Yt, 2))\n            rows.append([name, sum(q.numel() for q in m.parameters()), f\"{float((P == Yt).float().mean()):.3f}\", f\"{res[name][0]:.3f}\", f\"{res[name][1]:.3f}\"])\n        r = list(res.values())\n        ensure(r[1][1] > r[0][1] and r[2][1] > r[1][1] and r[3][1] > r[1][1] + 0.05, \"kenar doğruluğu: rastgele < bilineer < atlamalı / U-Net\")\n        ensure(r[3][0] > r[1][0] - 0.02, \"U-Net mIoU ≳ tek yukarı örneklemeli FCN (fark asıl kenarlarda)\")\n        R_.table(f\"64 × 64 sentetik sahneler, 1000 görüntü × 5 epoch ({time.perf_counter() - t00:.0f} sn)\", [\"model\", \"parametre\", \"piksel doğr.\", \"mIoU\", \"kenar doğr. (±2 px)\"], rows,\n                 note=\"Piksel doğruluğu arka planın çokluğu yüzünden yanıltıcı derecede yüksek; mIoU ve özellikle kenar doğruluğu farkı gösterir. Tek ×8 yukarı örnekleme sınırları bulanıklaştırır; sığ katmanlardan gelen bilgi (FCN-8s toplama, U-Net birleştirme) kenarları geri kazandırır. Bilineer başlatma, rastgele başlatılmış büyük tconv'un öğrenmesi gereken interpolasyonu hazır verir.\")\n    R_.test(\"FCN vs U-Net\", t_train)\n",
 "e": "import time\nimport torch\nfrom labcheck import need, ensure, fmt_time\n\n\ndef check(ns, R_):\n    def t():\n        torch.manual_seed(0)\n        U = need(ns, \"upsamplers\")(32)\n        ensure(set(U) == {\"tconv k3 s2\", \"tconv k4 s2\", \"nearest + conv3\", \"bilinear + conv3\", \"PixelShuffle\"}, \"beş yöntem\")\n        x = torch.randn(8, 32, 64, 64)\n        rows, art = [], {}\n        for name, m in U.items():\n            ensure(tuple(m(x[:1]).shape) == (1, 32, 128, 128), f\"{name}: ×2\")\n            with torch.no_grad():\n                m(x)\n                best = 1e9\n                for _ in range(3):\n                    t0 = time.perf_counter()\n                    m(x)\n                    best = min(best, time.perf_counter() - t0)\n            nparam = sum(p.numel() for p in m.parameters())\n            art[name] = need(ns, \"structural_artifact\")(m)\n            rows.append([name, nparam, fmt_time(best), f\"{art[name]:.3f}\"])\n        ensure(art[\"tconv k3 s2\"] > 0.5 and art[\"tconv k4 s2\"] < 1e-5 and art[\"nearest + conv3\"] < 1e-5, \"yapısal artefakt: yalnız k3 s2\")\n        R_.table(\"×2 yukarı örnekleme, 32 kanal, 8 × 64² → 128² (CPU)\", [\"yöntem\", \"parametre\", \"süre\", \"yapısal artefakt\"], rows,\n                 note=\"'Yapısal artefakt' = eşit ağırlıklar ve sabit girdiyle iç bölgede (en büyük − en küçük)/ortalama. k = 3, s = 2'lik tconv'da katkı sayısı 1-2-4 diye değiştiği için sabit girdi bile dama tahtası olur; k = 4 ve 'yeniden boyutlandır + konv.' düzgündür. PixelShuffle eşit ağırlıkla düzgündür ama rastgele başlatmada dört faz bağımsızdır (ICNR başlatması bunu giderir).\")\n    R_.test(\"yukarı örnekleyiciler\", t)\n",
 "x1": "import numpy as np\nfrom labcheck import close, max_loops, need, ensure\n\n\ndef _tconv(x, w, s):\n    Ci, H, W = x.shape\n    _, Co, k, _ = w.shape\n    out = np.zeros((Co, (H - 1) * s + k, (W - 1) * s + k))\n    for i in range(H):\n        for j in range(W):\n            out[:, i * s:i * s + k, j * s:j * s + k] += np.einsum(\"c,couv->ouv\", x[:, i, j], w)\n    return out\n\n\ndef check(ns, R_):\n    src = ns[\"__source__\"]\n    R_.test(\"döngüsüz\", lambda: [max_loops(src, 0, func=f) for f in (\"pixel_shuffle\", \"pixel_unshuffle\", \"subpixel_to_tconv\")])\n\n    def t():\n        rng = np.random.default_rng(0)\n        x = np.arange(2 * 4 * 3 * 3.).reshape(8, 3, 3)\n        y = need(ns, \"pixel_shuffle\")(x, 2)\n        ensure(y.shape == (2, 6, 6), \"şekil\")\n        ensure(y[1, 2 * 1 + 1, 2 * 2 + 0] == x[1 * 4 + 1 * 2 + 0, 1, 2], \"kanal c·r² + a·r + b → (h·r + a, w·r + b)\")\n        close(need(ns, \"pixel_unshuffle\")(y, 2), x, what=\"ters\")\n        for r in (2, 3):\n            Ci, Co = 5, 3\n            w = rng.normal(size=(Co * r * r, Ci))\n            xin = rng.normal(size=(Ci, 4, 5))\n            sub = ns[\"pixel_shuffle\"](np.einsum(\"oc,chw->ohw\", w, xin), r)\n            close(_tconv(xin, need(ns, \"subpixel_to_tconv\")(w, r), r), sub, atol=1e-10, what=f\"1×1 konv. + PixelShuffle = tconv(k = s = {r})\")\n        R_.note(\"Shi ve ark. (ESPCN): tüm konvolüsyonlar düşük çözünürlükte yapılır, en sonda kanallar uzaya 'açılır'. 1×1 durumda bu, k = s = r'lik transposed conv'un ta kendisidir (örtüşme yok → dama tahtası yok); k > 1'li alt-piksel konvolüsyonu, düşük çözünürlükte k, yüksekte r·k'lik bir tconv'a karşılık gelir ama r² kat daha ucuzdur.\")\n    R_.test(\"eşdeğerlik\", t)\n",
 "x2": "import numpy as np\nfrom labcheck import close, max_loops, need, ensure\n\n\ndef check(ns, R_):\n    src = ns[\"__source__\"]\n    R_.test(\"döngüsüz konvolüsyon\", lambda: max_loops(src, 0, func=\"dilated_conv2d\"))\n\n    def t():\n        rng = np.random.default_rng(0)\n        x = rng.normal(size=(12, 13))\n        w = rng.normal(size=(3, 3))\n        y = need(ns, \"dilated_conv2d\")(x, w, 2)\n        xp = np.pad(x, 2)\n        ref = sum(w[u, v] * xp[2 * u:2 * u + 12, 2 * v:2 * v + 13] for u in range(3) for v in range(3))\n        close(y, ref, atol=1e-10, what=\"d = 2\")\n        close(ns[\"dilated_conv2d\"](x, w, 1)[1:-1, 1:-1], sum(w[u, v] * x[u:u + 10, v:v + 11] for u in range(3) for v in range(3)), atol=1e-10, what=\"d = 1 normal konv.\")\n        ensure(need(ns, \"receptive_field\")([3] * 4, [1, 2, 4, 8]) == 31, \"1, 2, 4, 8 → 31\")\n        rows = []\n        for ds in ([1, 1, 1, 1], [2, 2, 2], [1, 2, 4, 8], [1, 2, 5], [1, 2, 3]):\n            Fp = need(ns, \"footprint\")(ds)\n            rows.append([str(ds), Fp.shape[0], int(Fp.sum()), f\"{Fp.mean():.2f}\"])\n        ensure(not ns[\"footprint\"]([2, 2, 2]).all() and ns[\"footprint\"]([1, 2, 5]).all(), \"ızgaralanma: [2,2,2] delikli, [1,2,5] tam\")\n        R_.table(\"3×3 genişletilmiş konvolüsyon yığınları\", [\"genişletmeler\", \"receptive field\", \"etkileyen piksel\", \"doluluk\"], rows,\n                 note=\"Genişletme, parametre ve çözünürlük kaybı olmadan receptive field'ı üstel büyütür (Yu & Koltun; DeepLab ASPP). Ama aynı d'yi tekrarlamak ızgaralanma (gridding) yaratır: merkez çıktı yalnız d'nin katlarındaki pikselleri görür. HDC (Wang ve ark.): ortak böleni 1 olan genişletmeler (1, 2, 5) alanı tam kaplar.\")\n    R_.test(\"dilated konvolüsyon\", t)\n",
 "x3": "import numpy as np\nfrom collections import deque\nfrom labcheck import need, ensure\n\n\ndef _bfs(mask):\n    H, W = mask.shape\n    lab = np.zeros((H, W), int)\n    K = 0\n    for i in range(H):\n        for j in range(W):\n            if mask[i, j] and not lab[i, j]:\n                K += 1\n                q = deque([(i, j)])\n                lab[i, j] = K\n                while q:\n                    a, b = q.popleft()\n                    for da, db in ((1, 0), (-1, 0), (0, 1), (0, -1)):\n                        x, y = a + da, b + db\n                        if 0 <= x < H and 0 <= y < W and mask[x, y] and not lab[x, y]:\n                            lab[x, y] = K\n                            q.append((x, y))\n    return lab, K\n\n\ndef check(ns, R_):\n    def t():\n        rng = np.random.default_rng(0)\n        for _ in range(5):\n            m = rng.random((30, 40)) < 0.45\n            lab, K = need(ns, \"connected_components\")(m)\n            ref, Kr = _bfs(m)\n            ensure(K == Kr and set(lab[~m].ravel()) <= {0}, \"bileşen sayısı\")\n            pairs = set(zip(lab[m].tolist(), ref[m].tolist()))\n            ensure(len(pairs) == K, \"bölümleme referansla aynı (etiket adları hariç)\")\n        sem = np.zeros((20, 30), int)\n        sem[2:8, 2:8] = 1\n        sem[2:8, 8:14] = 1\n        sem[12:18, 2:8] = 1\n        sem[12:18, 20:26] = 2\n        inst = need(ns, \"instances_from_semantic\")(sem, 3)\n        ensure(sorted((c, int(m.sum())) for c, m in inst) == [(1, 36), (1, 72), (2, 36)], \"örnekler\")\n        R_.note(\"Değen iki kare (sınıf 1) tek bir bileşen olarak çıkar: anlamsal maske, aynı sınıftan bitişik nesneleri ayırmaya yetmez. Örnek segmentasyonu bu yüzden ya tespitten başlar (Mask R-CNN: her kutu için ayrı maske) ya da ayırma sınırlarını ayrıca öğrenir (U-Net'in ağırlık haritası — ödev).\")\n    R_.test(\"bağlantılı bileşenler\", t)\n"
}

In [ ]:
import importlib, labcheck, checks_m4_10_segmentation as _c
importlib.reload(labcheck); importlib.reload(_c)
CHECKS = _c.CHECKS
print("hazır")

## Giriş

**Anlamsal segmentasyon** her pikseli ait olduğu nesnenin sınıfıyla etiketler (aynı sınıftan iki bisiklet tek bir leke olur); **örnek segmentasyonu** her nesneyi ayrıca ayırır. Zorluk: adımı 1'den büyük katmanlar çözünürlüğü kaybettirir — normal bir CNN "sol altta bir insan var" der ama sınırını bilmez.

**FCN (Long ve ark. 2015):** önceden eğitilmiş CNN'i FCN'e çevir (M4.9); toplam adım 32 olduğu için son haritalar 32 kat küçük → tek bir ×32 yukarı örnekleme katmanı (FCN-32s). Bilineer interpolasyon ×4–×8'e kadar iş görür; bunun yerine **transposed convolution**: girdiye boş satır/sütunlar (sıfırlar) ekleyip normal konvolüsyon yapmak — ya da kesirli adımlı (1/2) konvolüsyon. Bilineer interpolasyona yakın başlatılır ama öğrenilebilir. Burada adım, filtrenin adımı değil girdinin ne kadar gerileceğidir: büyük adım = büyük çıkış. Daha iyisi için **atlama bağlantıları**: ×2 büyüt, bir alt katmanın (aynı ölçek) çıkışını ekle, ×2 büyüt, daha alttakini ekle, ×8 büyüt (FCN-8s). Orijinal görüntüden büyük ölçeğe çıkmak **süper çözünürlüktür**.

**U-Net (2015):** tıbbi görüntü segmentasyonu için kodlayıcı-kod çözücü; kodlayıcıdaki her ölçeğin haritası, kod çözücüdeki aynı ölçeğe **kopyalanıp birleştirilir**; piksel sınıflandırması en son katmana bırakılır.

**Transposed conv'un dört yüzü:** (1) her girdi pikseli çekirdeği kendi değeriyle ölçekleyip çıkışa "saçar"; (2) sıfır ekleme + dolgu + ters çevrilmiş çekirdekle normal konvolüsyon; (3) konvolüsyonu bir matris $C$ olarak yazarsan $C^\top$; (4) konvolüsyonun girdisine göre gradyanı — bu yüzden bazen (yanlış olarak) "dekonvolüsyon" denir.

## 1 · Elle

<!-- çözüm -->

Çıkış: $(7-1)\cdot2-2\cdot1+4=14$.

1B scatter: 1 · [1, 2, 1] konum 0'dan, 2 · [1, 2, 1] konum 2'den, 3 · [1, 2, 1] konum 4'ten → [1, 2, 1 + 2, 4, 2 + 3, 6, 3] = [1, 2, 3, 4, 5, 6, 3]; indeks 4 = **5**. Çekirdek [½, 1, ½] olsaydı tam doğrusal interpolasyon olurdu.

Sıfır ekleme: [1, 0, 2, 0, 3] (uzunluk $(3-1)\cdot2+1=5$), her iki yana $k-1-p=2$ dolgu → 9; 3'lük konvolüsyonla 7 çıkış. ✓

Örtüşme (1B, k = 3, s = 2): çift konumlar 2, tek konumlar 1 katkı alır; 2B'de çarpım: 4, 2, 2, 1 → oran **4**. k = 4, s = 2'de her konum tam 2 (2B'de 4) katkı alır: düzgün.

Bilineer çekirdek (faktör 2): 1B $[0.25, 0.75, 0.75, 0.25]$, dış çarpım merkezi $0.75^2=0.5625$.

U-Net: 572 → 568 → 284 → 280 → 140 → 136 → 68 → 64 → 32 → 28 → (yukarı) 56 → 52 → 104 → 100 → 200 → 196 → 392 → **388**. Kodlayıcıdaki haritalar (568, 280, 136, 64) birleştirmeden önce kırpılır.

IoU $=4/(6+5-4)=4/7$; Dice $=2\cdot4/(6+5)=8/11$. Dice $=2\,\text{IoU}/(1+\text{IoU})$: aynı sıralamayı verir, küçük nesnelerde daha yumuşak.


Sitede bu aşama etkileşimli bir formdur. Burada değerleri aşağıdaki sözlüğe yazıp kontrol hücresini çalıştır.

**Transposed convolution**

- `osz`: Girdi 7 × 7, k = 4, s = 2, p = 1: çıkış kenarı
- `o4`: 1B: x = [1, 2, 3], w = [1, 2, 1], s = 2, p = 0 → çıkışın 4. indeksi (0'dan)
- `zl`: Aynı işlem 'sıfır ekle + normal konv.' olarak: sıfır eklenip dolgulanmış girdinin uzunluğu
- `cb`: 2B, k = 3, s = 2: iç bölgede bir çıkış pikseline katkı yapan girdi sayısının en büyüğü / en küçüğü
- `bil`: 2× bilineer yukarı örnekleme çekirdeği (k = 4) merkezdeki ağırlık

**U-Net ve ölçütler**

- `unet`: Özgün U-Net (valid 3×3 konv. ×2 her seviyede, 4 havuz, 2×2 yukarı konv.): 572 × 572 girdi → çıkış kenarı
- `iou`: Tahmin 6 piksel, gerçek 5 piksel, kesişim 4: IoU
- `dice`: Aynı örnek: Dice

In [ ]:
cevaplar = dict(
    osz=None,
    o4=None,
    zl=None,
    cb=None,
    bil=None,
    unet=None,
    iou=None,
    dice=None,
)

In [ ]:
#@title Kontrol (1)
_A = {"osz":[14,0],"o4":[5,0],"zl":[9,0],"cb":[4,0],"bil":[0.5625,1e-9],"unet":[388,0],"iou":[0.5714285714285714,0.000001],"dice":[0.7272727272727273,0.000001]}
_ok = 0
for k, (v, tol) in _A.items():
    g = cevaplar.get(k)
    good = g is not None and abs(float(g) - v) <= max(tol, 1e-9)
    _ok += good
    print(("✓ " if good else ("· " if g is None else "✗ ")) + k)
print(f"\n{_ok}/{len(_A)} doğru")

## 2 · Döngü (saf Python)

Saf Python.

- `tconv_out_size(H, k, s, p, op=0)`.
- `conv_transpose1d(x, w, s=1, p=0)` — scatter; tam çıkışı üretip baştan ve sondan `p` kırp.
- `conv_transpose2d(x, w, s=1, p=0)` — tek kanal, liste listeleri.
- `overlap_counts(n, k, s)` — `n` girdili 1B transposed conv'da her çıkış konumuna kaç girdinin katkı yaptığı.
- `unet_sizes(H, depth=4)` — valid U-Net: her seviyede iki 3×3 valid konv., 2×2 havuz; kod çözücüde 2× yukarı + iki valid konv. Ara boyutların listesi (girdi dahil, çıkış son eleman).
- `dice_iou(pred, gt)` — ikili maskeler (liste listeleri) → `(dice, iou)`; ikisi de boşsa (1, 1).

In [ ]:
%%writefile stage_b.py
def tconv_out_size(H, k, s, p, op=0):
    ...


def conv_transpose1d(x, w, s=1, p=0):
    ...


def conv_transpose2d(x, w, s=1, p=0):
    ...


def overlap_counts(n, k, s):
    ...


def unet_sizes(H, depth=4):
    ...


def dice_iou(pred, gt):
    ...


print(conv_transpose1d([1, 2, 3], [1, 2, 1], 2))

In [ ]:
# Kontrol (2): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_b.py", CHECKS["b"])

## 3 · NumPy / pandas

NumPy, döngüsüz. Ağırlık düzeni PyTorch'unki: `w (C_in, C_out, k, k)`.

- `conv_matrix(w, H, s=1, p=0)` — tek kanallı 2B konvolüsyonun (çapraz korelasyon, çekirdek `w (k, k)`, `H × H` girdi) matrisi $C$, şekil `(H_o², H²)`: `C @ x.ravel()` = konvolüsyon.
- `tconv2d(x, w, s=1, p=0)` — `x (C_in, H, W)`: scatter (`np.add.at` ile), sonra kırp.
- `tconv_zero_insert(x, w, s=1, p=0)` — sıfır ekle, `k − 1 − p` dolgula, kanal eksenlerini değiştirip çekirdeği 180° çevir, valid korelasyon.
- `bilinear_kernel(f)` — faktör `f` için `k = 2f − f mod 2` boyutlu 2B çekirdek (FCN başlatması).
- `confusion(pred, gt, C)` (`bincount`), `miou(cm)` → (sınıf IoU'ları, ortalama).

In [ ]:
%%writefile stage_c.py
import numpy as np
from numpy.lib.stride_tricks import sliding_window_view


def conv_matrix(w, H, s=1, p=0):
    ...


def tconv2d(x, w, s=1, p=0):
    ...


def tconv_zero_insert(x, w, s=1, p=0):
    ...


def bilinear_kernel(f):
    ...


def confusion(pred, gt, C):
    ...


def miou(cm):
    ...

In [ ]:
# Kontrol (3): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_c.py", CHECKS["c"])

## 4 · PyTorch

PyTorch. Veri: 64 × 64 tuvalde 2–5 şekil (kare, daire, eşkenar dörtgen; üst üste binebilir), 4 sınıf (0 = arka plan).

- `seg_dataset(n, seed)` — verilir.
- `conv_input_grad(y, w, s, p, H)` — `x ↦ conv2d(x, w)`'nin girdisine göre vektör-Jacobian çarpımı `vjp(y)` (autograd ile); `F.conv_transpose2d(y, w, stride=s, padding=p)` ile aynı olmalı.
- `bilinear_init_(tconv)` — `nn.ConvTranspose2d`'yi (C → C) kanal başına bilineer çekirdekle başlat.
- `FCN(C=4, w=12, skips=False, bilinear=True)` — paylaşılan kodlayıcı (`Encoder`, verilir; 3 havuz, adım 8); `skips=False`: 1×1 skor → ×8 transposed conv (k = 16, p = 4) ("FCN-8 adımlı-32s"); `skips=True`: ×2, + 1×1 skor(f3), ×2, + skor(f2), ×2 (FCN-8s düzeni).
- `UNet(C=4, w=12)` — her seviyede `ConvTranspose2d(k=2, s=2)` + birleştirme + iki Conv-BN-ReLU.
- `miou(pred, gt, C)` ve `edge_acc(pred, gt, r=2)` — etiket sınırına `r` piksel yakındaki piksellerde doğruluk (max/min havuzla).

In [ ]:
%%writefile stage_d.py
import torch
import torch.nn as nn
import torch.nn.functional as F


def seg_dataset(n, seed, size=64):
    g = torch.Generator().manual_seed(seed)
    yy, xx = torch.meshgrid(torch.arange(size) + 0.5, torch.arange(size) + 0.5, indexing="ij")
    X = torch.zeros(n, 1, size, size)
    Y = torch.zeros(n, size, size, dtype=torch.long)
    for i in range(n):
        for _ in range(int(torch.randint(2, 6, (1,), generator=g))):
            s = float(torch.randint(6, 24, (1,), generator=g))
            cx, cy = (torch.rand(2, generator=g) * size).tolist()
            c = int(torch.randint(0, 3, (1,), generator=g))
            if c == 0:
                m = ((xx - cx).abs() <= s / 2) & ((yy - cy).abs() <= s / 2)
            elif c == 1:
                m = (xx - cx) ** 2 + (yy - cy) ** 2 <= (s / 2) ** 2
            else:
                m = ((xx - cx).abs() + (yy - cy).abs()) <= s / 2
            X[i, 0][m] = 0.5 + 0.5 * float(torch.rand(1, generator=g))
            Y[i][m] = c + 1
    X += 0.15 * torch.randn(X.shape, generator=g)
    return X, Y


def cbr(a, b):
    return nn.Sequential(nn.Conv2d(a, b, 3, padding=1), nn.BatchNorm2d(b), nn.ReLU())


class Encoder(nn.Module):
    def __init__(self, w=12):
        super().__init__()
        self.e1 = nn.Sequential(cbr(1, w), cbr(w, w))
        self.e2 = nn.Sequential(cbr(w, 2 * w), cbr(2 * w, 2 * w))
        self.e3 = nn.Sequential(cbr(2 * w, 4 * w), cbr(4 * w, 4 * w))
        self.e4 = nn.Sequential(cbr(4 * w, 8 * w), cbr(8 * w, 8 * w))

    def forward(self, x):
        f1 = self.e1(x)
        f2 = self.e2(F.max_pool2d(f1, 2))
        f3 = self.e3(F.max_pool2d(f2, 2))
        f4 = self.e4(F.max_pool2d(f3, 2))
        return f1, f2, f3, f4


def conv_input_grad(y, w, s, p, H):
    ...


def bilinear_init_(tconv):
    ...


class FCN(nn.Module):
    def __init__(self, C=4, w=12, skips=False, bilinear=True):
        super().__init__()
        ...

    def forward(self, x):
        ...


class UNet(nn.Module):
    def __init__(self, C=4, w=12):
        super().__init__()
        ...

    def forward(self, x):
        ...


def miou(pred, gt, C):
    ...


def edge_acc(pred, gt, r=2):
    ...

In [ ]:
# Kontrol (4): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_d.py", CHECKS["d"])

## 5 · Vektörize + hız

- `upsamplers(C)` → sözlük (her biri `C → C`, ×2): `"tconv k3 s2"` (`output_padding=1`), `"tconv k4 s2"` (`padding=1`), `"nearest + conv3"`, `"bilinear + conv3"`, `"PixelShuffle"` (1×1 konv. C → 4C + `nn.PixelShuffle(2)`).
- `structural_artifact(m)` — tüm ağırlıklar eşit (1/eleman sayısı), bias 0, girdi sabit 1 iken iç bölgedeki çıkışın (en büyük − en küçük) / ortalama oranı.

In [ ]:
%%writefile stage_e.py
import time
import torch
import torch.nn as nn


def upsamplers(C):
    ...


def structural_artifact(m):
    ...

In [ ]:
# Kontrol (5): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_e.py", CHECKS["e"])

## Ek alıştırmalar

Beş aşamanın dışındaki pratikler; her biri kendi kontrolüyle.

### E1 · Alt-piksel konvolüsyonu (PixelShuffle) = transposed conv

NumPy.

- `pixel_shuffle(x, r)` — `(C r², H, W) → (C, H r, W r)`, kanal `c·r² + a·r + b` → konum `(h r + a, w r + b)`; `pixel_unshuffle(y, r)` tersi.
- `subpixel_to_tconv(w, r)` — 1×1 konvolüsyon ağırlığı `w (C_out r², C_in)` + PixelShuffle ile **aynı** sonucu veren transposed conv ağırlığı `(C_in, C_out, r, r)` (adım r).

In [ ]:
%%writefile extra_x1.py
import numpy as np


def pixel_shuffle(x, r):
    ...


def pixel_unshuffle(y, r):
    ...


def subpixel_to_tconv(w, r):
    ...

In [ ]:
# Kontrol (E1)
labcheck.run_file("extra_x1.py", CHECKS["x1"])

### E2 · Genişletilmiş (dilated) konvolüsyon ve ızgaralanma

NumPy.

- `dilated_conv2d(x, w, d)` — tek kanal, "same" dolgu, genişletme `d`.
- `receptive_field(ks, ds)` — adım 1'li yığın: $1+\sum(k_i-1)d_i$.
- `footprint(ds, k=3)` — 3×3 genişletilmiş konvolüsyonlar yığınının merkez çıkışına etki eden girdi piksellerinin ikili haritası (desteklerin ardışık "Minkowski toplamı").

In [ ]:
%%writefile extra_x2.py
import numpy as np


def dilated_conv2d(x, w, d):
    ...


def receptive_field(ks, ds):
    ...


def footprint(ds, k=3):
    ...

In [ ]:
# Kontrol (E2)
labcheck.run_file("extra_x2.py", CHECKS["x2"])

### E3 · Anlamsaldan örneğe: bağlantılı bileşenler

NumPy (iterasyon döngüsü serbest).

- `connected_components(mask)` — 4-komşuluk; etiket yayılımı: her ön plan pikseline benzersiz bir sayı ver, değişmeyene kadar her pikseli kendisi ve ön plandaki 4 komşusunun en küçüğüyle değiştir; sonra etiketleri 1..K'ye yeniden numarala. `(etiketler, K)`.
- `instances_from_semantic(sem, C)` — her sınıf için ayrı bileşenler → `[(sınıf, maske)]`.

In [ ]:
%%writefile extra_x3.py
import numpy as np


def connected_components(mask):
    ...


def instances_from_semantic(sem, C):
    ...

In [ ]:
# Kontrol (E3)
labcheck.run_file("extra_x3.py", CHECKS["x3"])

## Makale ödevi: Long, Shelhamer, Darrell (2015): FCN-32s/16s/8s

[Long, Shelhamer, Darrell — Fully Convolutional Networks for Semantic Segmentation (CVPR 2015)](https://arxiv.org/abs/1411.4038)

Önceden eğitilmiş bir VGG-16'nın yoğun katmanlarını konvolüsyona çevir (fc6 → 7 × 7 konv., fc7 → 1 × 1; ağırlıkları yeniden şekillendirerek, M4.9 x1), PASCAL VOC 2012'nin küçük bir alt kümesinde FCN-32s, ardından 16s ve 8s'i makaledeki aşamalı ince ayar düzeninde eğit; Tablo 2'deki gibi piksel doğruluğu, ortalama doğruluk, mIoU ve frekans ağırlıklı IoU raporla. Yukarı örnekleme katmanlarını bilineer başlat; donuk bilineer ile öğrenilebiliri karşılaştır.


**Adımlar**

1. c aşamasındaki bilinear_kernel ve confusion/miou'yu kullan.

**Kabul ölçütleri**

- FCN-8s, FCN-32s'ten mIoU'da belirgin iyi.
- Öğrenilebilir yukarı örnekleme ile donuk bilineer arasındaki fark küçük (makaleyle uyumlu).

## Makale ödevi: Ronneberger ve ark. (2015): U-Net ve ağırlık haritası

[Ronneberger, Fischer, Brox — U-Net: Convolutional Networks for Biomedical Image Segmentation (MICCAI 2015)](https://arxiv.org/abs/1505.04597)

Valid konvolüsyonlu özgün U-Net'i (572 → 388) ve döşemeli (overlap-tile) çıkarımı uygula; sentetik "birbirine değen hücreler" veri setinde (x3'teki gibi) makaledeki Denk. (2) ağırlık haritasıyla — $w(x)=w_c(x)+w_0\,e^{-(d_1+d_2)^2/2\sigma^2}$, $w_0=10$, $\sigma\approx5$ — ve onsuz eğit. Ayırma çizgilerinin doğru öğrenildiğini örnek sayısı (bağlantılı bileşen) üzerinden ölç; elastik deformasyon artırmasını ekle.


**Adımlar**

1. $d_1$, $d_2$: her piksel için en yakın ve ikinci en yakın hücrenin sınırına uzaklık.

**Kabul ölçütleri**

- Ağırlık haritasıyla değen hücreler ayrı örnekler olarak çıkar; onsuz birleşir.
- Döşemeli çıkarım, tek parça çıkarımla kenarlarda aynı sonucu verir.

## Kaynaklar

- [Géron — Hands-On Machine Learning (3. baskı), Bölüm 14](https://github.com/ageron/handson-ml3) — Apache-2.0 (kod)
- [Long, Shelhamer, Darrell — Fully Convolutional Networks for Semantic Segmentation (CVPR 2015)](https://arxiv.org/abs/1411.4038) — arXiv
- [Ronneberger, Fischer, Brox — U-Net (MICCAI 2015)](https://arxiv.org/abs/1505.04597) — arXiv
- [Dumoulin & Visin — A guide to convolution arithmetic for deep learning](https://arxiv.org/abs/1603.07285) — arXiv
- [Odena, Dumoulin, Olah — Deconvolution and Checkerboard Artifacts (Distill 2016)](https://distill.pub/2016/deconv-checkerboard/) — CC-BY 4.0
- [Shi ve ark. — Real-Time Single Image and Video Super-Resolution Using an Efficient Sub-Pixel CNN (CVPR 2016)](https://arxiv.org/abs/1609.05158) — arXiv
- [Yu & Koltun — Multi-Scale Context Aggregation by Dilated Convolutions (ICLR 2016)](https://arxiv.org/abs/1511.07122) — arXiv
- [Wang ve ark. — Understanding Convolution for Semantic Segmentation (HDC, WACV 2018)](https://arxiv.org/abs/1702.08502) — arXiv
- [d2l.ai — 14.10 Transposed Convolution, 14.11 FCN](https://d2l.ai/chapter_computer-vision/transposed-conv.html) — CC-BY-SA 4.0